In [ ]:
from adler.objectdata.AdlerPlanetoid import AdlerPlanetoid
from adler.science.PhaseCurve import PhaseCurve
import adler.utilities.science_utilities as sci_utils
from adler.utilities.plotting_utilities import plot_errorbar

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import astropy.units as u
import yaml
from astropy.io.misc.yaml import load

In [ ]:
# ssObjectId of object to analyse
ssoid1 = "21164741220119127"  # test object

In [ ]:
# here we use an offline SQL database which contains the observations of the sso
fname = "../gen_test_data/adler_demo_testing_database_dp2.db"
planetoid1 = AdlerPlanetoid.construct_from_SQL(
    "21164741220119127", sql_filename=fname, schema="dp2", flux_flag="psfFlux"
)

# # or query directly when on RSP
# planetoid1 = AdlerPlanetoid.construct_from_RSP(ssoid1, schema="dp1", flux_flag="psfFlux")

In [ ]:
planetoid1

In [ ]:
planetoid1.__dict__

In [ ]:
planetoid1.MPCORB

In [ ]:
planetoid1.SSObject

In [ ]:
planetoid1.filter_list

In [ ]:
fig = plot_errorbar(
    planetoid1, filt_list=planetoid1.filter_list, x_plot="midpointMjdTai", y_plot="reduced_mag"
)
fig = plot_errorbar(
    planetoid1,
    filt_list=planetoid1.filter_list,
    x_plot="phaseAngle",
    y_plot="reduced_mag",
    label_list=planetoid1.filter_list,
)
plt.gca().legend()
plt.title(planetoid1.MPCORB.fullDesignation)
# # TODO: deal with negative uncertainties?
# fig = plot_errorbar(planetoid1, filt_list=planetoid1.filter_list, x_plot="midpointMjdTai", y_plot="reduced_mag", yerr_plot = None)
# fig = plot_errorbar(planetoid1, filt_list=planetoid1.filter_list, x_plot="phaseAngle", y_plot="reduced_mag", yerr_plot = None)

In [ ]:
# inspect observations

In [ ]:
filt = "r"
obs = planetoid1.observations_in_filter(filt)

In [ ]:
df_obs = pd.DataFrame(obs.__dict__)

In [ ]:
df_obs

In [ ]:
tmin = np.amin(np.floor(df_obs["midpointMjdTai"]))  # mjd
tmax = np.amax(np.floor(df_obs["midpointMjdTai"])) + 1  # mjd
tmin, tmax

In [ ]:
# cumulative data in filter
x_plot = "midpointMjdTai"
df_plot = df_obs.sort_values(x_plot)

fig = plt.figure()
gs = gridspec.GridSpec(1, 1)
ax1 = plt.subplot(gs[0, 0])

bins = np.arange(tmin, tmax + 1)

values, base = np.histogram(df_plot[x_plot], bins=bins)
cumulative = np.cumsum(values)
ax1.plot(base[:-1] - base[0], cumulative, label=filt)

data_mask = np.diff(cumulative) > 0
data_nights = base[1:-1][data_mask]
N_data = cumulative[1:][data_mask]

ax1.scatter(data_nights - base[0], N_data)

ax1.set_xlabel(x_plot)
ax1.set_ylabel("number")
ax1.legend()

plt.show()

In [ ]:
# number of data points per night of new data
np.diff(N_data)

In [ ]:
# nights when new data arrives
data_nights